# YOLO Training Pipeline - ESP32 Optimized

**Optimasi untuk ESP32-CAM:**
1. Auto-Annotation dari folder
2. Dataset Balancing (150/class)
3. Train/Val/Test Split (70:20:10)
4. Enhanced Augmentations (flip, brightness, rotation, blur)
5. 100 Epochs dengan Early Stopping
6. ONNX Export untuk Deployment

## Step 1: Setup & Configuration

In [ ]:
!pip install ultralytics opencv-python numpy matplotlib seaborn -q

In [ ]:
import os
import json
import cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
import random
import matplotlib.pyplot as plt
import seaborn as sns

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Configuration
BASE_DIR = Path(r'c:\Users\Sanzz\OneDrive\Documents\my_dataset')
DATASET_SOURCE = BASE_DIR / 'dataset'
ANNOTATIONS_FILE = BASE_DIR / 'annotations' / 'annotations.json'
OUTPUT_DIR = BASE_DIR / 'yolo_dataset'
MODEL_OUTPUT = BASE_DIR / 'yolo_output'

# ESP32 Optimized Parameters
TARGET_COUNT_PER_CLASS = 150
TRAIN_RATIO = 0.70  # 70% train
VAL_RATIO = 0.20    # 20% validation
TEST_RATIO = 0.10   # 10% test
IMG_SIZE = 96       # Small for ESP32-CAM
EPOCHS = 100        # More epochs for convergence
BATCH_SIZE = 16

CLASS_NAMES = ['dinding', 'kendaraan_parkir', 'orang', 'pintu', 'pohon', 'tiang']
CLASS_COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']

print(f'Classes: {CLASS_NAMES}')
print(f'Split Ratio - Train: {TRAIN_RATIO*100:.0f}%, Val: {VAL_RATIO*100:.0f}%, Test: {TEST_RATIO*100:.0f}%')
print(f'Image Size: {IMG_SIZE}x{IMG_SIZE}')
print(f'Epochs: {EPOCHS}')

## Step 2: Auto-Annotation

In [ ]:
def create_auto_annotations():
    '''Create annotations from folder structure (center-based 60%)'''
    annotations = {}
    
    for class_id, class_name in enumerate(CLASS_NAMES):
        class_folder = DATASET_SOURCE / class_name
        if not class_folder.exists():
            print(f'Warning: {class_folder} not found')
            continue
        
        count = 0
        for img_file in class_folder.glob('*'):
            if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.webp']:
                try:
                    img = cv2.imread(str(img_file))
                    if img is None:
                        continue
                    h, w = img.shape[:2]
                    
                    annotations[str(img_file)] = {
                        'class_id': class_id,
                        'class_name': class_name,
                        'bbox': [0.2, 0.2, 0.8, 0.8],
                        'width': w,
                        'height': h
                    }
                    count += 1
                except:
                    pass
        print(f'  {class_name}: {count} images')
    
    ann_dir = BASE_DIR / 'annotations'
    ann_dir.mkdir(exist_ok=True)
    with open(ann_dir / 'annotations.json', 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f'Total: {len(annotations)} annotations')
    return annotations

if ANNOTATIONS_FILE.exists():
    print('Loading existing annotations...')
    with open(ANNOTATIONS_FILE, 'r') as f:
        annotations = json.load(f)
    print(f'Loaded {len(annotations)} annotations')
else:
    print('Creating auto-annotations...')
    annotations = create_auto_annotations()

## Step 3: Dataset Balancing + Enhanced Augmentation

In [ ]:
def convert_bbox_to_yolo(bbox):
    x_min, y_min, x_max, y_max = bbox
    cx = (x_min + x_max) / 2
    cy = (y_min + y_max) / 2
    w = x_max - x_min
    h = y_max - y_min
    return max(0.001, min(0.999, cx)), max(0.001, min(0.999, cy)), max(0.01, min(1.0, w)), max(0.01, min(1.0, h))

def augment_image(img, aug_type):
    '''Enhanced augmentations: hflip, bright, dark, rotation, blur'''
    if aug_type == 0:  # Horizontal flip
        return cv2.flip(img, 1), 'hflip', True
    elif aug_type == 1:  # Brightness increase
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,2] = np.clip(hsv[:,:,2] * 1.2, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR), 'bright', False
    elif aug_type == 2:  # Brightness decrease
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:,:,2] = np.clip(hsv[:,:,2] * 0.8, 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR), 'dark', False
    elif aug_type == 3:  # Rotation
        angle = random.uniform(-10, 10)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        return cv2.warpAffine(img, M, (w, h)), 'rot', False
    elif aug_type == 4:  # Blur
        return cv2.GaussianBlur(img, (3, 3), 0), 'blur', False
    else:
        return img, 'orig', False

def flip_bbox(bbox):
    return [1 - bbox[2], bbox[1], 1 - bbox[0], bbox[3]]

def get_split(rand_val):
    if rand_val < TRAIN_RATIO:
        return 'train'
    elif rand_val < TRAIN_RATIO + VAL_RATIO:
        return 'val'
    else:
        return 'test'

print('Augmentation types: hflip, bright, dark, rotation, blur')

In [ ]:
# Group by class and balance
class_items = defaultdict(list)
for img_path, ann in annotations.items():
    class_items[ann['class_id']].append({
        'path': img_path,
        'bbox': ann['bbox'],
        'class_id': ann['class_id'],
        'class_name': ann['class_name']
    })

print(f'Balancing (target: {TARGET_COUNT_PER_CLASS}/class)...')
balanced = {}

for class_id, items in class_items.items():
    current = len(items)
    
    if current >= TARGET_COUNT_PER_CLASS:
        balanced[class_id] = random.sample(items, TARGET_COUNT_PER_CLASS)
        action = 'undersample'
    else:
        balanced[class_id] = items.copy()
        need = TARGET_COUNT_PER_CLASS - current
        for i in range(need):
            item = random.choice(items).copy()
            item['augment'] = i % 5
            item['aug_id'] = i
            balanced[class_id].append(item)
        action = 'oversample'
    
    print(f'  {CLASS_NAMES[class_id]}: {current} -> {len(balanced[class_id])} ({action})')

In [ ]:
# Create directories and process
for split in ['train', 'val', 'test']:
    (OUTPUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

print('Processing images...')
total = 0
stats = {'train': 0, 'val': 0, 'test': 0}

for class_id, items in balanced.items():
    for item in items:
        try:
            img = cv2.imread(item['path'])
            if img is None:
                continue
            
            bbox = item['bbox']
            aug_suffix = ''
            
            if 'augment' in item:
                img, aug_name, need_flip = augment_image(img, item['augment'])
                aug_suffix = f'_aug{item["aug_id"]}_{aug_name}'
                if need_flip:
                    bbox = flip_bbox(bbox)
            
            cx, cy, bw, bh = convert_bbox_to_yolo(bbox)
            split = get_split(random.random())
            
            base_name = Path(item['path']).stem
            new_name = f'{CLASS_NAMES[class_id]}_{base_name}{aug_suffix}'
            
            cv2.imwrite(str(OUTPUT_DIR / 'images' / split / f'{new_name}.jpg'), img)
            
            with open(OUTPUT_DIR / 'labels' / split / f'{new_name}.txt', 'w') as f:
                f.write(f'{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
            
            total += 1
            stats[split] += 1
        except:
            pass
    print(f'  Processed {CLASS_NAMES[class_id]}')

print(f'\nTotal: {total} images')
print(f'Train: {stats["train"]} | Val: {stats["val"]} | Test: {stats["test"]}')

In [ ]:
# Create dataset.yaml
yaml_content = f'''path: {OUTPUT_DIR}
train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
'''
with open(OUTPUT_DIR / 'dataset.yaml', 'w') as f:
    f.write(yaml_content)

print('Dataset YAML created!')

## Step 4: Training YOLOv5n (100 Epochs + Early Stopping)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov5n.pt')

results = model.train(
    data=str(OUTPUT_DIR / 'dataset.yaml'),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=str(MODEL_OUTPUT),
    name='yolo_training',
    exist_ok=True,
    patience=20,
    save=True,
    plots=True
)

## Step 5: Evaluate on Test Set

In [ ]:
metrics = model.val(
    data=str(OUTPUT_DIR / 'dataset.yaml'),
    split='test',
    imgsz=IMG_SIZE
)

print(f'\n=== TEST RESULTS ===')
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

## Step 6: Export to ONNX for ESP32

In [ ]:
model.export(format='onnx', imgsz=IMG_SIZE, simplify=True)

print('\n' + '='*60)
print('EXPORT COMPLETE!')
print('='*60)
print(f'Model: {MODEL_OUTPUT}/yolo_training/weights/best.onnx')